# I. Importing Data

In [ ]:
# Import th? vi?n 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6) 
import warnings
warnings.simplefilter("ignore")


Ch?ng ta ??c ri?ng d? li?u t? th?ng 1 n?m 2015 ??n th?ng 4 n?m 2016 v? th?ng 5 n?m 2016. H?n n?a, v? df kh? l?n n?n ch?ng ta gi?i h?n k?ch th??c b?ng c?ch l?y m?u 20% unique ID v? l?c c? df v?i danh s?ch ID k?t qu?.

In [ ]:
train = pd.read_csv("/kaggle/input/dataflow2025-product-recommendation/SPLITDATASET/train.csv",
                dtype={"sexo":str, "ind_nuevo":str, 
                                 "ult_fec_cli_1t":str, 
                                  "indext":str})
test = pd.read_csv("/kaggle/input/dataflow2025-product-recommendation/SPLITDATASET/test.csv",
                  dtype={"sexo":str, "ind_nuevo":str, 
                                 "ult_fec_cli_1t":str, 
                                  "indext":str})

In [ ]:
unique_ids = pd.Series(train["ncodpers"].unique())
train = train[train.ncodpers.isin(unique_ids)]
test = test[test.ncodpers.isin(unique_ids)]

In [ ]:
print(f'?? d?i c?a t?p train: {len(train)}')
print(f'?? d?i c?a t?p test: {len(test)}')

In [ ]:
unique_id = unique_ids.sample(frac = 0.2)   
train = train[train.ncodpers.isin(unique_id)]
test = test[test.ncodpers.isin(unique_id)]

In [ ]:
train.describe()

In [ ]:
test.describe()

# Data Cleaning

In [ ]:
# Ch?ng ta c? t?ng c?ng bao nhi?u b?n ghi ?
print("Ch?ng ta c? s? b?ng ghi l?:", train["ncodpers"].count())

In [ ]:
train.dtypes

In [ ]:
test.dtypes

In [ ]:
# Thay ??i ki?u d? li?u 
train["age"] = pd.to_numeric(train["age"], errors="coerce") 
train["antiguedad"] = pd.to_numeric(train["antiguedad"], errors="coerce") 
train["indrel_1mes"] = pd.to_numeric(train["indrel_1mes"], errors="coerce") 

In [ ]:
test["age"] = pd.to_numeric(test["age"], errors="coerce") 
test["antiguedad"] = pd.to_numeric(test["antiguedad"], errors="coerce") 
test["indrel_1mes"] = pd.to_numeric(test["indrel_1mes"], errors="coerce") 
test["renta"] = pd.to_numeric(test["renta"], errors="coerce") 

In [ ]:
# Gi? tr? b? thi?u v?i t?ng c?t 
train.isnull().sum()

In [ ]:
# Gi? tr? b? thi?u v?i t?ng c?t 
test.isnull().sum()

In [ ]:
# X?a c?t 
train = train.drop(["ult_fec_cli_1t", "conyuemp"], axis=1) 

In [ ]:
# X?a c?t 
test = test.drop(["ult_fec_cli_1t", "conyuemp"], axis=1) 

In [ ]:
#Ta s? thay th? nh?ng gi? tr? b? thi?u ? c?t renta b?ng tthu nh?p trung b?nh c?a kh?ch h?ng trong c?ng m?t t?nh 
grouped = train.groupby("nomprov").agg({"renta":lambda x: x.median(skipna=True)}).reset_index()
new_incomes = pd.merge(train,grouped,how="inner",on="nomprov").loc[:, ["nomprov","renta_y"]]
new_incomes = new_incomes.rename(columns={"renta_y":"renta"}).sort_values("renta").sort_values("nomprov")
train.sort_values("nomprov",inplace=True)
train = train.reset_index()
new_incomes = new_incomes.reset_index()

train.loc[train.renta.isnull(),"renta"] = new_incomes.loc[train.renta.isnull(),"renta"].reset_index()
train.loc[train.renta.isnull(),"renta"] = train.loc[train.renta.notnull(),"renta"].median()
train.sort_values(by="fecha_dato",inplace=True)

In [ ]:
grouped = test.groupby("nomprov").agg({"renta":lambda x: x.median(skipna=True)}).reset_index()
new_incomes = pd.merge(test,grouped,how="inner",on="nomprov").loc[:, ["nomprov","renta_y"]]
new_incomes = new_incomes.rename(columns={"renta_y":"renta"}).sort_values("renta").sort_values("nomprov")
test.sort_values("nomprov",inplace=True)
test = test.reset_index()
new_incomes = new_incomes.reset_index()

test.loc[train.renta.isnull(),"renta"] = new_incomes.loc[test.renta.isnull(),"renta"].reset_index()
test.loc[train.renta.isnull(),"renta"] = test.loc[test.renta.notnull(),"renta"].median()
test.sort_values(by="fecha_dato",inplace=True)

In [ ]:
train = train.dropna(axis=0)

In [ ]:
test = test.dropna(axis=0)

In [ ]:
#C? m?t s? kh?ch h?ng c? th?m ni?n ?t h?n 0. Ta s? thay th? b?ng 0
train[train["antiguedad"]<0] = 0

In [ ]:
test[test["antiguedad"]<0] = 0

In [ ]:
# Th?m m?t c?t m?i v? t?ng s? s?n ph?m c?a m?i kh?ch h?ng m?i th?ng
train["tot_products"] = train.loc[:,"ind_ahor_fin_ult1":"ind_recibo_ult1"].sum(axis=1)
train["tot_products"] = pd.to_numeric(train["tot_products"], errors="coerce") 

In [ ]:
#B?y gi? ch?ng ta s? ki?m tra ph?n ph?i kh?ch h?ng theo qu?c gia
train['pais_residencia'].describe() 

> Ta c? th? th?y top qu?c gia l? "ES". N?n ta s? lo?i b? b?t c?c qu?c gi? kh?c ?? c? m?t t?p d? li?u t?p trung.

In [ ]:
train = train.loc[lambda train: train.pais_residencia == "ES", :]

In [ ]:
test = test.loc[lambda test: test.pais_residencia == "ES", :]

In [ ]:
#Ta s? ??m th?m ch? s? nh?n vi?n
train['ind_empleado'].value_counts()

> Ta th?y r?ng nh?n vi?n c? ch? s? 'N' l? nhi?u nh?t, cho n?n ta t?p trung v?o n?

In [ ]:
train = train.loc[lambda train: train.ind_empleado == "N", :]

In [ ]:
test = test.loc[lambda test: test.ind_empleado == "N", :]

C?c d? li?u ?? ???c l?m s?ch, v?y cho n?n ta s? ??n ph?n ti?p theo 

# Feature Engineering

In [ ]:
#Ta s? copy df ra m?t b?n m?i
train_b = train.copy()

?? x?c ??nh t?t h?n nh?ng thay ??i trong danh m?c s?n ph?m theo t?ng kh?ch h?ng, ta th?m m?t t?nh n?ng ?? hi?n th? th?ng quan s?t (t?c l? t?p d? li?u b?t ??u v?o th?ng 1 n?m 2015 l? th?ng 0 v? th?ng 2 n?m 2015 l? th?ng 1)

In [ ]:
months = list(set(train_b["fecha_dato"]))
months.sort()

def date_to_month(date):
  month = months.index(date)
  return month

train_b["Month"] = train_b["fecha_dato"].apply(date_to_month)
train_b["next_Month"] = train_b["Month"] + 1

## Features

C?c t?nh n?ng hi?n c?
?? s? d?ng ???c nhi?u lo?i danh m?c kh?c nhau trong m? h?nh ?? ch?n, ta ph?i m? h?a c?c danh m?c hai chi?u th?nh c?c bi?n nh? ph?n v? m? h?a m?t l?n c?c danh m?c ?a chi?u th?nh c?c bi?n nh? ph?n do ?nh h??ng c?a ch?ng ??i v?i c? hai khung d? li?u (th?ng 1 n?m 2015 - th?ng 4 n?m 2016, th?ng 5 n?m 2016)

C?c bi?n sau s? b? b? qua trong m? h?nh:
- fecha_dato: Thay v? ng?y th?c t?,ta t?p trung v?o th?ng c?a b?n ghi, feature ???c t?o ? b??c tr??c ??.
- ind_empleado: Do gi? tr? c? ?? bi?n thi?n c?c th?p n?n ta t?p trung v?o c?c b?n ghi c? gi? tr? chi?m ?u th? (t?c l? "N")
- pais_residencia: V? ta ch? t?p trung v?o kh?ch h?ng T?y Ban Nha (nh? ?? t?m th?y trong qu? tr?nh d?n d?p d? li?u), n?n bi?n n?y l? th?a
- fecha_alta: Thay v? ng?y th?c t?, ta s? s? d?ng th?m ni?n t?nh b?ng th?ng
- indresi: V? qu?c gia c? tr? c?a ph?n l?n kh?ch h?ng gi?ng v?i qu?c gia c?a ng?n h?ng ('S'), n?n ta ch? t?p trung v?o S v? bi?n n?y l? th?a
   Ta lo?i b? c?c c?t s?n ph?m
- ind_aval_fin_ult1 v? ind_ahor_fin_ult1 v? nh? ?? th?ytrong ph?n kh?m ph? v? tr?c quan h?a d? li?u 2 c?i n?y r?t ?t

In [ ]:
#X?a c?c t?nh n?ng kh?ng c?n thi?t
train_b.drop(["fecha_dato", "ind_empleado", "pais_residencia", 'fecha_alta','indresi','ind_aval_fin_ult1','ind_ahor_fin_ult1' ], axis = 1, inplace = True)

#Chuy?n ki?u d? li?u th?nh ki?u category 
train_b["ind_nuevo"] = train_b["ind_nuevo"].astype(float).astype(int)
to_be_coded = ["sexo", "tiprel_1mes", "indext", "canal_entrada", "indfall", "nomprov", "segmento"]
for i in to_be_coded:
    train_b[i] = train_b[i].astype('category') 
    train_b[i] = train_b[i].cat.codes

In [ ]:
#X?a c?c t?nh n?ng c?a df_may t??ng t? 
test.drop(["fecha_dato", "ind_empleado", "pais_residencia", 'fecha_alta','indresi'], axis = 1, inplace = True)

#Chuy?n ki?u d? li?u th?nh category
test["ind_nuevo"] = test["ind_nuevo"].astype(float).astype(int)  

to_be_coded = ["sexo", "tiprel_1mes", "indext", "canal_entrada", "indfall", "nomprov", "segmento"]
for i in to_be_coded:
    test[i] = test[i].astype('category') 
    test[i] = test[i].cat.codes

In [ ]:
#T?o t?n c?t cho t?ng s?n ph?m 
labels = train_b.columns
products = list(filter(lambda k: 'ult' in k, labels))

In [ ]:
len(products)

## New Features 
B? d? li?u hi?n t?i c?a ta bao g?m th?ng tin v? t?ng kh?ch h?ng v? c?c s?n ph?m ?ang ho?t ??ng c?a h?. M?c ti?u c?a m? h?nh l? t?m ra s?n ph?m n?o m? kh?ch h?ng c? kh? n?ng mua nhi?u nh?t. Do ??, s? c? l?i n?u ``t?p trung v?o s? thay ??i c?a s?n ph?m ??i v?i t?ng kh?ch h?ng v? t?ng th?ng``. ??i v?i ?i?u n?y, ta s? t?o th?m c?c t?nh n?ng ?? tr? (lag features) cho t?ng s?n ph?m v? t?ng th?ng, m? t?:

1. S?n ph?m ?? ho?t ??ng trong bao l?u trong nh?ng th?ng tr??c? (product_cumsum):
T?nh n?ng n?y t?m t?t c?c th?ng s?n ph?m ?? ho?t ??ng tr??c th?ng c?a b?n ghi ???c chu?n h?a theo s? th?ng cho ??n b?n ghi.V? d?: n?u s?n ph?m ?? ho?t ??ng trong th?i gian d?i v?i kh?ch h?ng, th? t?c l? kh?ch h?ng th?ch s?n ph?m ??y. B? qua l? do ngo?i l? khi?n s?n ph?m ng?ng b?n, n?i chung l? t?nh n?ng n?y s? t?nh ???c kh? n?ng s?n ph?m ?? c? ???c kh?ch h?ng ch?n l?i hay kh?ng

3. Quy?n s? h?u s?n ph?m ?? thay ??i nh? th? n?o so v?i th?ng tr??c (product_change):
T?nh n?ng n?y cho bi?t s?n ph?m ?? b? lo?i b? (-1) hay ???c th?m v?o (1) ho?c kh?ng c? g? x?y ra v?i s?n ph?m ?? (0) so v?i th?ng tr??c. ? ??y, ch?ng ta c? th? cho r?ng n?u m?t s?n ph?m ch? m?i b? lo?i b? c?ch ??y m?t th?ng, th? kh? n?ng s?n ph?m ?? ???c ch?n l?i c? th? th?p h?n so v?i s?n ph?m kh?c


5. C? bao nhi?u giao d?ch (?? th?m v? ?? lo?i b?) li?n quan ??n s?n ph?m (product_transaction):
T?nh n?ng n?y t?ng h?p t?t c? c?c giao d?ch cho t?ng s?n ph?m cho ??n th?ng ghi ch?p. N?u m?t s?n ph?m ?? ???c th?m v? lo?i b? nhi?u l?n, th? kh? n?ng s?n ph?m ?? ???c th?m l?i c? th? cao h?n.

In [ ]:
# 1.S?n ph?m ?? ho?t ??ng trong bao l?u trong nh?ng th?ng tr??c 

#M?c ti?u l? t?o c?t cho m?i s?n ph?m,  t?m t?t th?i gian ho?t ??ng c?a s?n ph?m trong th?ng tr??c cho t?ng kh?ch h?ng

#??u ti?n ta t?o b?n sao c?a b?n b?ng c?ch lo?i b? t?t c? c?c t?nh n?ng(features) kh?ng c?n thi?t cho ho?t ??ng n?y
train_c = train_b.copy()
df_products_only = pd.concat([train_c[["ncodpers", "Month"]], train_c.filter(regex = "ult" )], axis=1)
df_products_only.sort_values(by = "Month", inplace = True)

#B?y gi? ta t?o m?t DF nh?m theo cho m?i th?ng ?? t?ng h?p c?c th?ng ho?t ??ng v? n?i c?c k?t qu? v?i nhau
df_product_sum = pd.DataFrame()
for i in set(df_products_only["Month"]):
    zz = df_products_only[df_products_only["ncodpers"].isin(df_products_only[df_products_only["Month"] == i]["ncodpers"])]
    z = zz[zz["Month"] < i+1].drop("Month", axis = 1).groupby(by = ["ncodpers"]).sum().reset_index()    
    z["Month"] = i
    df_product_sum = pd.concat([df_product_sum, z])
    
#Sau ??, ch?ng ta chu?n h?a c?c c?t b?ng c?ch chia n? cho s? th?ng
for i in products:
  df_product_sum[i] = df_product_sum[i]/df_product_sum["Month"]

#V? ta chia cho O ?? ghi l?i trong th?ng ??u ti?n n?n ta ph?i thay NaN b?ng 0
df_product_sum.fillna(0, inplace = True)
df_product_sum.replace([np.inf, -np.inf], 0, inplace=True)

#B?y gi? ta m? r?ng featureDf ban ??u c?a m?nh b?ng c?ch h?p nh?t n? v?i feature n?y
df_features= train_b.merge(df_product_sum, on = ["ncodpers", "Month"], suffixes = ["", "_cumsum"])

In [ ]:
# 2. Vi?c s? h?u s?n ph?m ?? thay ??i nh? th? n?o so v?i th?ng tr??c

## ??u ti?n, ta t?o hai b?n sao c?a df c?a ch?ng ta, lo?i b? t?t c? c?c t?nh n?ng(features) kh?ng c?n thi?t cho ho?t ??ng n?y. 
train_d = train_b.copy()  #df ch?a m?t c?t b? sung v?i th?ng tr??c ?? ???c s? d?ng ?? m? r?ng b?n ghi v?i c?c b?n ghi c?a th?ng tr??c ??
df_products_only = pd.concat([train_d[["ncodpers", "Month"]], train_d.filter(regex = "ult" )], axis=1)
df_products_only2 = df_products_only.copy()
df_products_only["prev_Month"] = df_products_only["Month"] -1

#G?p c? hai khung d? li?u v?i nhau ?? t?o m?t df c? th?ng tin v? th?ng hi?n t?i v? th?ng tr??c:
df_change = df_products_only.merge(df_products_only2, left_on = ["ncodpers", "prev_Month"], right_on = ["ncodpers", "Month"], suffixes = ["", "_previous"], how = "left")

## T?nh to?n s? thay ??i c?a s?n ph?m b?ng c?ch tr? tr?ng th?i s?n ph?m hi?n t?i v?i tr?ng th?i s?n ph?m
for i in products:
  current_prod = i 
  prev_prod = i + "_previous"
  change = i + "_change"
  df_change[change] = df_change[current_prod] - df_change[prev_prod]
  df_change[change].astype(int, errors = "ignore")

## ?i?n 0 v?o NaN v? kh?ng c? thay ??i n?o trong Th?ng 0
df_change.fillna(0, inplace = True)

#M?t s? m? h?nh c? th? b? l?i v?i c?c gi? tr? ?m v? v?y ta th?m 1 v?o t?t c? c?c gi? tr?
df_change = df_change + 1

# H?p nh?t df thay ??i v?i df t?nh n?ng (feature df) 
df_features2 = pd.concat([df_features, df_change.filter(regex = "change" )], axis=1)

In [ ]:
# 3.C? bao nhi?u giao d?ch ?? t?n t?i li?n quan ??n s?n ph?m

# M?c ti?u l? t?o m?t c?t hi?n th? s? l?n s?n ph?m ???c th?m v?o ho?c x?a cho ??n th?i ?i?m ghi ch?p
# ??u ti?n,ta thay ??i gi? tr? c?a change_columns t? 0 th?nh 1 n?u ph?t sinh giao d?ch 
def change_to_transaction(x):
    if x != 0:
        y = 1
    else:
        y = 0
    return y
df_transaction = pd.DataFrame()

for i in df_change.filter(regex = "change" ).columns.values:
    j = i[:-7] + "_transaction"
    df_transaction[j] = df_change.filter(regex = "change" )[i].apply(change_to_transaction)
df_transaction = pd.concat([df_features2[["ncodpers", "Month"]], df_transaction], axis=1)

#B?y gi? ta t?o m?t s? DF cho m?i th?ng ?? t?ng h?p s? l??ng giao d?ch cho ??n th?ng t??ng ?ng v? n?i c?c k?t qu? v?i nhau
df_transaction_sum = pd.DataFrame()
for i in set(df_transaction["Month"]):
    z = df_transaction[df_transaction["Month"] < i+1].drop("Month", axis = 1).groupby(by = ["ncodpers"]).sum().reset_index()    
    z["Month"] = i
    df_transaction_sum = pd.concat([df_transaction_sum, z])

#Cu?i c?ng, ta h?p nh?t n? v?i t?nh n?ng m?i nh?t c?a ta l? DF
df_features3 = df_features2.merge(df_transaction_sum, on = ["ncodpers", "Month"], suffixes = ["", "_cumsum"])

In [ ]:
len(df_features3)

## Target Variables 

C?c bi?n m?c ti?u c?a ta cho bi?t li?u s?n ph?m c? ???c th?m v?o hay kh?ng

In [ ]:
df_two_months = train_b.merge(train_b, left_on = ["ncodpers", "next_Month"], right_on = ["ncodpers", "Month"], suffixes = ["_actual", "_next"])
y_status =df_two_months.filter(regex='ult1_actual')

In [ ]:
#M?c ti?u l? t?o m?t df v?i th?ng tin s? ???c th?m v?o trong th?ng t?i

#??u ti?n, ta h?p nh?t DF g?c v?i ch?nh n? ?? th?m th?ng tin s?n ph?m c?a th?ng ti?p theo v?o m?i b?n ghi

df_two_months = train_b.merge(train_b, left_on = ["ncodpers", "next_Month"], right_on = ["ncodpers", "Month"], suffixes = ["_actual", "next"])

# T?nh to?n s? thay ??i c?a s?n ph?m b?ng c?ch tr? tr?ng th?i s?n ph?m c?a th?ng ti?p theo v?i tr?ng th?i s?n ph?m c?a th?ng hi?n t?i
for i in products:
  next_prod = i + "next"
  actual_prod = i + "_actual"
  change = i + "_change"
  df_two_months[change] = df_two_months[next_prod] - df_two_months[actual_prod]
  df_two_months[change].astype(int, errors = "ignore")

# B?y gi? ta x?a t?t c? c?c c?t ngo?i tr? nh?ng c?t c? s? thay ??i
df_change2 = df_two_months.filter(regex='change')

#Sau ??, ta thay ??i gi? tr? c?a c?c c?t m?c ??ch l? ?? hi?n th? r?ng s?n ph?m ?? ???c th?m hay ch?a 
def activate (x):
  if x == 1:
    y = 1
  else:
    y = 0
  return y

iterCol = df_change2.columns.values

for i in iterCol:
  colName = i[:-7]+"_activated"
  df_change2[colName] = df_change2[i].apply(activate)


y_activated = df_change2.filter(regex='_activated')

## X v? Y 

In [ ]:
#??u ti?n ta x?a c?c t?nh n?ng(features) t? th?ng tr??c , v? ta s? s? d?ng ch?ng ?? d? ?o?n c?c ?? xu?t c?a m?nh t? th?ng55
df_features_june_prediction = df_features3[df_features["Month"] == df_features3["Month"].max()]
df_features_training = df_features3[df_features3["Month"] < df_features3["Month"].max()]

#B?y x?a c?c h?ng c?n l?i kh?ng c? bi?n m?c ti?u:
df_features_training = df_features_training[df_features_training["index"].isin(df_two_months["index_actual"])]


#B?y gi? h?y b? ??nh danh ?? ch?ng ta ch? c? c?c t?nh n?ng(features)
X_may = df_features_june_prediction.drop(["index", "Month","next_Month"], axis = 1)
X = df_features_training.drop(["index","ncodpers", "Month","next_Month"], axis = 1)
     

> Ch?ng t?i c?ng ph?i h?p nh?t th?ng tin c?a c?c s?n ph?m t? th?ng 4 n?m 2016 v?i khung d? li?u c?a th?ng 5 n?m 2016 ?? d? ?o?n s?n ph?m s? ???c ?? xu?t cho th?ng 5 n?m 2016.

In [ ]:
#X_may ch?a th?ng tin t? th?ng 4 nh?ng ch? d?nh cho th?ng tin s?n ph?m
#??i v?i th?ng kh?ch h?ng, ta s? d?ng th?ng tin c? s?n t? th?ng 5 n?m 2016 (t?c l? test ? tr?n)
#Do ??, ta ph?i x?a ch?ng kh?i X_may v? h?p nh?t X_may ?? gi?m v?i df_may
to_drop = ['sexo', 'age', 'ind_nuevo', 'antiguedad', 'indrel',
       'indrel_1mes', 'tiprel_1mes', 'indext', 'canal_entrada', 'indfall',
       'tipodom', 'cod_prov', 'nomprov', 'ind_actividad_cliente', 'renta',
       'segmento']

X_may.drop(to_drop, inplace = True, axis = 1)

X_may = test.merge(X_may, on = "ncodpers", how = "inner")
index = X_may.ncodpers
X_may.drop(["ncodpers", "index"], inplace = True, axis = 1)

Ta ch? quan t?m ??n c?c m?c c? giao d?ch (t?c l? m?t s?n ph?m ???c th?m v?o) trong m?t th?ng nh?t ??nh, v?:
- Giao d?ch l? m?t vi?c t??ng ??i hi?m. Do ??, r?t nhi?u d? li?u c?n th?a th?t li?n quan ??n giao d?ch
- H?nh vi c?a kh?ch h?ng c? th? ph? thu?c v?o th?i ?i?m trong n?m v?i xu h??ng theo m?a. Do ?? ta s? ??o t?o m?t m? h?nh cho m?t th?ng nh?t ??nh ch?   v?i d? li?u t? th?ng ?? t? nh?ng n?m tr??c

In [ ]:
def specify_df(transaction_only = True, month = ""):
  if transaction_only == True:
    sum = y_activated.sum(axis=1)
    if month != "":
      y_only_transaction = y_activated[(sum > 0) & (dfb["Month"] == month)]
      x_only_transaction = X[(sum > 0) & (dfb["Month"] == month)]
    else:
      y_only_transaction = y_activated[(sum > 0)]
      X1 = X.reset_index(drop = True)
      x_only_transaction = X1[(sum > 0)]
  else:
    y_only_transaction = y_activated[dfb["Month"] == month]
    x_only_transaction = X[dfb["Month"] == month]

  return x_only_transaction, y_only_transaction

#applying it
x1,y1 = specify_df(True)

# Nh?p 

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x1, y1, random_state=42)

In [ ]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [ ]:
#classifying multi-label data with multioutputclassifier & xgboost with default parameters
xgboost_classifier = MultiOutputClassifier(estimator=XGBClassifier())
xgboost_classifier.fit(X_train, y_train)

In [ ]:
y_train_pred = xgboost_classifier.predict(X_train)
y_train_pred = pd.DataFrame(y_train_pred)

In [ ]:
y_pred = xgboost_classifier.predict(X_test)
y_pred = pd.DataFrame(y_pred)

In [ ]:
#compute accuracy+precision metrics on training set --> for data scientists 
#lets create a df to see how well the model performs for each class = product 
scores = pd.DataFrame(columns = ['Product','Accuracy','Precision'])
scores["Product"] = products

for i in range(22):
  scores["Precision"][i] = precision_score(y_train.iloc[:,i],y_train_pred.iloc[:,i])

for i in range(22):
  scores["Accuracy"][i] = accuracy_score(y_train.iloc[:,i],y_train_pred.iloc[:,i])

#scores

#overall
print("Overall train_accuracy across all products: ", scores.Accuracy.mean().round(3))
print("Overall train_precision across all products: ", scores.Precision.mean().round(3))

In [ ]:
#as info for data scientists only 
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names = products))

In [ ]:
def average_precision_at_k(y_true, y_pred, k=10):
    """
    T?nh AP@K (Average Precision at K)
    
    Parameters:
    - y_true: Danh s?ch c?c labels th?c t? (1: relevant, 0: not relevant)
    - y_pred: Danh s?ch d? ?o?n ???c s?p x?p theo ?? li?n quan gi?m d?n
    - k: Top-K ???c x?t ??n
    
    Returns:
    - AP@K score
    """
    if len(y_pred) > k:
        y_pred = y_pred[:k]

    num_relevant = 0
    score = 0.0

    for i, pred in enumerate(y_pred):
        if pred in y_true:
            num_relevant += 1
            score += num_relevant / (i + 1)

    return score / min(len(y_true), k) if y_true else 0.0

def mean_average_precision_at_k(y_true_list, y_pred_list, k=10):
    """
    T?nh MAP@K (Mean Average Precision at K)
    
    Parameters:
    - y_true_list: Danh s?ch danh s?ch c?c labels th?c t? cho m?i truy v?n
    - y_pred_list: Danh s?ch danh s?ch d? ?o?n s?p x?p theo ?? li?n quan
    - k: Top-K ???c x?t ??n
    
    Returns:
    - MAP@K score
    """
    ap_scores = [average_precision_at_k(y_true, y_pred, k) for y_true, y_pred in zip(y_true_list, y_pred_list)]
    return sum(ap_scores) / len(ap_scores) if ap_scores else 0.0

# V? d? s? d?ng:
y_true_example = [[1, 2, 3], [2, 4]]
y_pred_example = [[1, 3, 5, 2], [4, 2, 3, 1]]

map_score = mean_average_precision_at_k(y_true_example, y_pred_example, k=3)
print(f"MAP@3: {map_score:.4f}")
